h(t) = A * h(t-1) + B * x(t)     # update the hidden state
y(t) = C * h(t)                  # produce an output

h is the hidden state ("running summary" of the sequence so far)
x is the input @ each step
y is the output

A, B, C are matrices that control the behavior


In [41]:
import torch
import torch.nn as nn

test code to ensure algorithm works

In [42]:
input_dim = 4   # size of x(t)
hidden_dim = 8  # size of h(t)
output_dim = 4  # size of y(t)

A = torch.randn(hidden_dim, hidden_dim)  # hidden -> hidden
B = torch.randn(hidden_dim, input_dim)   # input -> hidden
C = torch.randn(output_dim, hidden_dim)  # hidden -> output

def ssm_step(x, h_prev):
    h = A @ h_prev + B @ x  # h(t) = A * h(t-1) + B * x(t)
    y = C @ h               # y(t) = C * h(t)
    return h, y

seq_len = 10
x_seq = torch.randn(seq_len, input_dim)
h = torch.zeros(hidden_dim)
outputs = []

for t in range(seq_len):
    h, y = ssm_step(x_seq[t], h)
    outputs.append(y)

In [43]:
class SelectiveSSM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.A = nn.Parameter(-torch.abs(torch.randn(hidden_dim)))
        self.linear_B = nn.Linear(input_dim, hidden_dim)
        self.hidden_dim = hidden_dim
        self.linear_delta = nn.Linear(input_dim, hidden_dim)

    def forward(self, x_seq):
        batch_size, seq_len, _ = x_seq.shape
        h = torch.zeros(batch_size, self.hidden_dim)
        outputs = []

        for t in range(seq_len):
            x_t = x_seq[:, t, :]
            B_t = self.linear_B(x_t)
            delta = nn.functional.softplus(self.linear_delta(x_t))
            A_bar = torch.exp(delta * self.A)
            B_bar = delta * B_t
            h = A_bar * h + B_bar
            outputs.append(h)

        return torch.stack(outputs, dim=1)

first test

In [44]:
# generate random sequences of integers as dummy data
batch_size = 32
seq_len = 10
vocab_size = 16

x = torch.randint(0, vocab_size, (batch_size, seq_len))
y = x.clone()  # target is identical to input

print(x[0])
print(y[0])

tensor([ 4,  6,  5,  5,  5, 13, 15,  1, 10,  0])
tensor([ 4,  6,  5,  5,  5, 13, 15,  1, 10,  0])


In [45]:
class MambaModel(nn.Module):
    def __init__(self, vocab_size, input_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, input_dim)
        self.ssm = SelectiveSSM(input_dim, hidden_dim)
        self.output_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)       # integers -> vectors
        h_states = self.ssm(x)      # run through SSM
        logits = self.output_head(h_states)  # vectors -> predictions
        return logits

In [46]:
model = MambaModel(vocab_size=16, input_dim=32, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

for step in range(1000):
    data = torch.randint(0, vocab_size, (batch_size, seq_len + 1))
    x = data[:, :-1]   # everything except the last token
    y = data[:, 1:]    # everything except the first token 
    
    logits = model(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

Step 0, Loss: 2.8570432662963867
Step 100, Loss: 2.7961370944976807
Step 200, Loss: 2.78482985496521
Step 300, Loss: 2.784418821334839
Step 400, Loss: 2.781550645828247
Step 500, Loss: 2.7723708152770996
Step 600, Loss: 2.7793428897857666
Step 700, Loss: 2.7693123817443848
Step 800, Loss: 2.7700889110565186
Step 900, Loss: 2.7703754901885986


In [48]:
!pip install datasets

from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-v1")

  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.10.2-py3-none-any.whl.metadata (14 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached huggingface_hub-1.10.2-py3-none-any.whl (642 kB)
Using cached hf_xet-1.4.3-cp37-abi3-macosx_11_0_arm64.whl (3.6 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 47.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.0
    Uninstalling pyarrow-19.0.0:
      Successfully uninstalled pyarrow-19.0.0
  Attempting uninstall: dill━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [pyarrow]
    Found existing installation: dill 0.3.8━━━━━━━━━━━━━━━━━━━ 1/7 [pyarrow]
    Uninstalling dill-0.3.8:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [pyarrow]
      Successfu

README.md: 0.00B [00:00, ?B/s]

wikitext-2-v1/test-00000-of-00001.parque(…):   0%|          | 0.00/685k [00:00<?, ?B/s]

wikitext-2-v1/train-00000-of-00001.parqu(…):   0%|          | 0.00/6.07M [00:00<?, ?B/s]

wikitext-2-v1/validation-00000-of-00001.(…):   0%|          | 0.00/618k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [49]:
print(dataset['train'][0])
print(dataset['train'][1])
print(dataset['train'][2])

{'text': ''}
{'text': ' = Valkyria Chronicles III = \n'}
{'text': ''}


In [ ]:
!pip install transformers

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 48.5 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2024.11.6
    Uninstalling regex-2024.11.6:
      Successfully uninstalled regex-2024.11.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [52]:
# join all the text into one long string
full_text = "\n".join(dataset['train']['text'])

# tokenize it
tokens = tokenizer.encode(full_text)
print(f"Total tokens: {len(tokens)}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"First 20 tokens: {tokens[:20]}")

Token indices sequence length is longer than the specified maximum sequence length for this model (2434240 > 1024). Running this sequence through the model will result in indexing errors


Total tokens: 2434240
Vocab size: 50257
First 20 tokens: [198, 796, 569, 18354, 7496, 17740, 6711, 796, 220, 628, 198, 2311, 73, 13090, 645, 569, 18354, 7496, 513, 1058]
